# SasRec with PR-Entmax

PR-Entmax is a sparse, prior-regularized objective designed to reduce popularity bias while preserving recommendation relevance. For $\alpha > 1$, the loss can produce sparse logit gradients, while the item-frequency prior provides popularity correction during training. No architectural changes are required; at inference, the prior correction is omitted and items are ranked by contextual relevance alone.

Paper: https://arxiv.org/pdf/2610.17823

$$
p_i^*(z) =
\left[
(\alpha - 1)
\left(
\frac{z_i + \gamma \log q_i}{\theta} - \tau
\right)
\right]_+^{\frac{1}{\alpha - 1}}.
$$

Here, $q_i$ is the normalized item-frequency prior, $\theta$ is the temperature, and $\tau$ ensures that probabilities sum to one; $[x]_+ = \max(x, 0)$.

We recommend setting $\alpha = 1.5$, which performed best in our experiments and is also used in RePlay. We suggest starting with $\theta = 10$ and tuning $\gamma$ in the range $[0.1, 1.0]$ to control the strength of the prior correction.

Data preparation and the standard training pipeline follow [the SasRec example](09_sasrec_example.ipynb); this notebook focuses on configuring PR-Entmax and comparing it with CE.

**To improve performance, set ARROW_NUM_THREADS and ARROW_IO_THREADS to the number of available CPUs. Set OMP_NUM_THREADS, MKL_NUM_THREADS, torch num_threads, and num_interop_threads to 1.**

In [1]:
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env ARROW_NUM_THREADS=8
%env ARROW_IO_THREADS=8

env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: ARROW_NUM_THREADS=8
env: ARROW_IO_THREADS=8


In [2]:
import lightning as L  
import pandas as pd
import torch

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

L.seed_everything(42);

Seed set to 42


## Preparing data
In this example, we will use the MovieLens dataset, namely the 1M subset. A simple case is demonstrated, so only item ids will be used as a model input.


In [3]:
interactions = pd.read_csv("./data/ml1m_ratings.dat", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
interactions = interactions.drop(columns=["rating"])

In [4]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions

,user_id,item_id,timestamp
1000138,6040,858,0
999873,6040,593,1
1000153,6040,2384,2
1000007,6040,1961,3
1000192,6040,2019,4
...,...,...,...
825793,4958,2399,446
825438,4958,1407,447
825731,4958,2634,448
825724,4958,3264,449


### Encode categorical data
To ensure all categorical data is fit for training, it needs to be encoded using the `LabelEncoder` class. Create an instance of the encoder, providing a `LabelEncodingRule` for each categorical column in the dataset that will be used in a model. Note that ids of users and ids of items are always used.

In [5]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)
encoded_interactions = encoder.fit_transform(interactions)
encoded_interactions

,timestamp,user_id,item_id
0,0,6039,802
1,1,6039,579
2,2,6039,2191
3,3,6039,1781
4,4,6039,1839
...,...,...,...
1000204,446,4957,2206
1000205,447,4957,1305
1000206,448,4957,2432
1000207,449,4957,3040


### Split interactions into the train, validation and test datasets using LastNSplitter
We use a widespread splitting strategy Last-One-Out. We filter out cold items and users for simplicity.

In [6]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True,
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

### Dataset preprocessing ("baking")
SasRec expects each user in the batch to provide their events in a form of a sequence. For this reason, the event splits must be properly processed using the `groupby_sequences` function provided by RePlay.

In [7]:
from replay.data.nn.utils import groupby_sequences


def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")

In [8]:
train_events = bake_data(train_events)

validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)

test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events

,user_id,timestamp,item_id
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[2969, 957, 1574, 1178, 2147, 1658, 3177, 1117..."
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1108, 1127, 1120, 2512, 1201, 2735, 1135, 110..."
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[579, 2651, 1788, 3301, 1781, 1327, 1174, 3429..."
3,3,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1120, 1025, 466, 3235, 3294, 1106, 1108, 253,..."
4,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[847, 2512, 858, 346, 1158, 2651, 2007, 1050, ..."
...,...,...,...
6035,6035,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1574, 3206, 2183, 1703, 2235, 2651, 2501, 249..."
6036,6036,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1702, 1848, 1175, 3275, 672, 2932, 802, 548, ..."
6037,6037,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[859, 3165, 1120, 1965, 346, 1288, 1007, 2511,..."
6038,6038,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[275, 107, 1886, 869, 1139, 886, 2917, 2809, 2..."


To ensure we don't have unknown users in ground truth, we join validation events and validation ground truth (also join test events and test ground truth correspondingly) by user ids to leave only the common ones.  

In [9]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [10]:
from pathlib import Path

data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
PREDICT_PATH = data_dir / "test.parquet"

ENCODER_PATH = data_dir / "encoder"

In [11]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(PREDICT_PATH)

encoder.save(ENCODER_PATH)

/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/replay/preprocessing/label_encoder.py:1002: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/replay/preprocessing/label_encoder.py:563: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


## Prepare for model training
### Create the tensor schema
A schema shows the correspondence of columns from the source dataset to the internal representation of tensors inside the model. It is required by the NN models to correctly create embeddings for every source column. Note that user_id is not required in `TensorSchema`.

Note that **cardinality** is the number of unique values ​in the item catalog (vocabulary). **Padding value** is the next value after the last one.

In [12]:
from replay.data import FeatureHint, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorSchema

EMBEDDING_DIM = 64

NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)

### Compute the item-frequency prior

Estimate $q_i$ as the number of training interactions with item $i$ divided by the total number of training interactions. Use training data only and align the vector with encoded item IDs; items absent from training receive zero frequency. The resulting vector is passed to `PREntmax` as `item_frequency` and used for popularity-aware logit adjustment during training.

In [13]:
# Count training interactions per item, aligned with encoded item IDs.
item_counts = train_events["item_id"].explode().value_counts().reindex(range(NUM_UNIQUE_ITEMS), fill_value=0)
# q_i = item interaction count / total training interactions.
item_frequency = torch.tensor((item_counts / item_counts.sum()).to_numpy(), dtype=torch.float32)

### Configure ParquetModule and transformation pipelines

In [14]:
from replay.nn.transform.template import make_default_sasrec_transforms

transforms = make_default_sasrec_transforms(tensor_schema)

In [15]:
MAX_SEQ_LEN = 50

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": tensor_schema["item_id"].padding_value},
    },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
        "ground_truth": {"shape": 1, "padding": -1},
    },
}

In [16]:
from replay.data.nn import ParquetModule

BATCH_SIZE = 32

## Train and compare models

### SasRec configuration

Below, we configure the number of transformer blocks, attention heads, and dropout for the shared SasRec architecture.
The `create_model(loss)` function is used for both training and checkpoint loading, so CE and PR-Entmax differ only in the loss.

In [17]:
from replay.nn.agg import SumAggregator
from replay.nn.embedding import SequenceEmbedding
from replay.nn.loss import CE, PREntmax
from replay.nn.mask import DefaultAttentionMask
from replay.nn.sequential import PositionAwareAggregator, SasRec, SasRecBody, SasRecTransformerLayer

NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3


def create_model(loss):
    excluded_features = list({tensor_schema.query_id_feature_name, tensor_schema.timestamp_feature_name})
    body = SasRecBody(
        embedder=SequenceEmbedding(
            schema=tensor_schema,
            categorical_list_feature_aggregation_method="sum",
            excluded_features=excluded_features,
        ),
        embedding_aggregator=PositionAwareAggregator(
            embedding_aggregator=SumAggregator(embedding_dim=EMBEDDING_DIM),
            max_sequence_length=MAX_SEQ_LEN,
            dropout=DROPOUT,
        ),
        attn_mask_builder=DefaultAttentionMask(
            reference_feature_name=tensor_schema.item_id_feature_name,
            num_heads=NUM_HEADS,
        ),
        encoder=SasRecTransformerLayer(
            embedding_dim=EMBEDDING_DIM,
            num_heads=NUM_HEADS,
            num_blocks=NUM_BLOCKS,
            dropout=DROPOUT,
            activation="relu",
            hidden_dim=None,
        ),
        output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
    )
    # Attach the selected loss without changing the SasRec architecture.
    return SasRec(body=body, loss=loss)

### Configure PR-Entmax

Pass the training prior to PREntmax through the item_frequency argument, and set the correction strength with gamma and the temperature.

In [18]:
from functools import partial

from replay.nn.lightning import LightningModule

loss_factories = {
    "CE": partial(CE, ignore_index=NUM_UNIQUE_ITEMS),
    # Pass the training prior q to PR-Entmax.
    "PR-Entmax": partial(
        PREntmax,
        item_frequency=item_frequency,
        gamma=0.2, 
        temperature=10,
        ignore_index=NUM_UNIQUE_ITEMS,
    ),
}

### Train each model

The train_model function resets the seed before creating the model and creates fresh data modules and callbacks. Both runs train for 100 epochs, selecting the best checkpoint by validation Recall@10. Checkpoints and CSV logs are stored separately for each loss.

In [19]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from replay.nn.lightning.callback import ComputeMetricsCallback

TOPK = [10, 20]


def train_model(name, loss_factory):
    print(f"Training {name}...", flush=True)

    L.seed_everything(42)
    model = LightningModule(create_model(loss_factory()))
    data_module = ParquetModule(
        train_path=TRAIN_PATH,
        validate_path=VAL_PATH,
        batch_size=BATCH_SIZE,
        metadata=train_metadata,
        transforms=transforms,
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath=Path("sasrec/checkpoints") / name,
        save_top_k=1,
        verbose=False,
        monitor="recall@10",
        mode="max",
    )
    validation_metrics_callback = ComputeMetricsCallback(
        metrics=["ndcg", "recall"],
        ks=TOPK,
        item_count=NUM_UNIQUE_ITEMS,
        verbose=False,
    )

    trainer = L.Trainer(
        max_epochs=100,
        devices=1,
        callbacks=[checkpoint_callback, validation_metrics_callback],
        logger=CSVLogger(save_dir="sasrec/logs/train", name=name),
        enable_progress_bar=False,
    )
    trainer.fit(model, datamodule=data_module)
    checkpoint_path = checkpoint_callback.best_model_path
    best_epoch = int(checkpoint_path.split("epoch=")[1].split("-")[0])
    validation_metrics = validation_metrics_callback.get_metrics()[best_epoch]

    print(
        f"{name}: best epoch = {best_epoch + 1}, "
        f"validation Recall@10 = {validation_metrics['recall@10']:.6f}",
        flush=True,
    )
    return {
        "checkpoint_path": checkpoint_path,
        "best_epoch": best_epoch,
        "validation_metrics": validation_metrics,
    }

Run both experiments and inspect the validation metrics from each best checkpoint.

The table shows the best validation epoch for each loss, numbered from 1.

In [20]:
training_results = {name: train_model(name, loss_factory) for name, loss_factory in loss_factories.items()}

Training CE...


Seed set to 42
/tmp/ipykernel_3982851/1103087989.py:14: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  data_module = ParquetModule(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_c

CE: best epoch = 27, validation Recall@10 = 0.211790
Training PR-Entmax...


Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jovyan/home/jovyan/Pembek/sasrec/checkpoints/PR-Entmax exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name  | Type   | Params | Mode  | FLOPs
-------------------------------------------------
0 | model | SasRec | 291 K  | train | 0    
-------------------------------------------------
291 K     Trainable params
0         Non-trainable params
291 K     Total params
1.164     Total estimated model params size (MB)
40        Modules in train mode
0         Modules in eval mode
0         Total Flops
`Tra

PR-Entmax: best epoch = 95, validation Recall@10 = 0.223050


In [21]:
pd.DataFrame(
    [
        {
            "loss": name,
            "best_epoch": result["best_epoch"] + 1,
            **result["validation_metrics"],
        }
        for name, result in training_results.items()
    ]
).set_index("loss")

,best_epoch,recall@10,ndcg@10,recall@20,ndcg@20
loss,,,,,
CE,27,0.21179,0.095899,0.328034,0.125085
PR-Entmax,95,0.22305,0.104750,0.329359,0.131456


## Inference

Reconstruct each model with create_model and load its best checkpoint. PR-Entmax uses the item-frequency prior only during training; at inference, both models rank items using raw model logits.

In [22]:
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

In [23]:
from replay.nn.lightning.callback import PandasTopItemsCallback


def predict_model(name, loss_factory, checkpoint_path):
    best_model = LightningModule.load_from_checkpoint(
        checkpoint_path,
        model=create_model(loss_factory()),
    )
    best_model.eval()
    data_module = ParquetModule(
        predict_path=PREDICT_PATH,
        batch_size=BATCH_SIZE,
        metadata=inference_metadata,
        transforms=transforms,
    )
    callback = PandasTopItemsCallback(
        top_k=max(TOPK),
        query_column="user_id",
        item_column="item_id",
        rating_column="score",
    )
    trainer = L.Trainer(
        devices=1,
        callbacks=[callback],
        logger=CSVLogger(save_dir="sasrec/logs/test", name=name),
        inference_mode=True,
        enable_progress_bar=False,
    )
    trainer.predict(best_model, datamodule=data_module, return_predictions=False)
    return callback.get_result()

Generate recommendations for the same test users with both best checkpoints. PandasTopItemsCallback returns encoded item IDs and their model scores.

In [24]:
predictions = {
    name: predict_model(name, loss_factory, training_results[name]["checkpoint_path"])
    for name, loss_factory in loss_factories.items()
}

/tmp/ipykernel_3982851/3305662044.py:10: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  data_module = ParquetModule(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers 

In [25]:
predictions["PR-Entmax"]

,user_id,item_id,score
0,0,740,4.278793
0,0,1439,4.030858
0,0,47,3.965979
0,0,1551,3.504745
0,0,354,3.48153
...,...,...,...
6037,6039,225,0.80164
6037,6039,1536,0.800495
6037,6039,521,0.772146
6037,6039,1478,0.770405


### Compare CE and PR-Entmax

Compare recommendation quality using Recall and NDCG, and catalog diversity and novelty using Coverage and Surprisal. Coverage measures how much of the training catalog appears in recommendations; Surprisal measures item rarity based on unique training users.

Both models use the same architecture, data splits, seed, and training budget, with checkpoints selected by validation Recall@10.

In [26]:
from replay.metrics import NDCG, Coverage, Experiment, Recall, Surprisal

In [27]:
ground_truth = test_gt[["user_id", "item_id"]].explode("item_id")
metric_train = train_events[["user_id", "item_id"]].explode("item_id")

experiment = Experiment(
    metrics=[Recall(TOPK), NDCG(TOPK), Coverage(TOPK), Surprisal(TOPK)],
    ground_truth=ground_truth,
    train=metric_train,
    query_column="user_id",
    rating_column="score",
)
for name, recommendations in predictions.items():
    experiment.add_result(name, recommendations)

experiment.results

,Recall@10,Recall@20,NDCG@10,NDCG@20,Coverage@10,Coverage@20,Surprisal@10,Surprisal@20
CE,0.19311,0.303577,0.089218,0.116944,0.566028,0.651094,0.249987,0.256798
PR-Entmax,0.20106,0.305399,0.098046,0.124260,0.652174,0.776668,0.267754,0.282769


#### Relative changes compared with CE

For this configuration, PR-Entmax improves relevance (NDCG by 6–10% and Recall by up to 4%) while reducing popularity bias (Coverage by 15–19% and Surprisal by 7–10%) compared with CE.

In [28]:
relative_changes = 100 * (
    experiment.results.loc[["PR-Entmax"]] / experiment.results.loc["CE"] - 1
)
relative_changes.style.format("{:+.2f}%")

,Recall@10,Recall@20,NDCG@10,NDCG@20,Coverage@10,Coverage@20,Surprisal@10,Surprisal@20
PR-Entmax,+4.12%,+0.60%,+9.90%,+6.26%,+15.22%,+19.29%,+7.11%,+10.11%
